# 01.8 · Statistics you need, by hand

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/08-statistics-by-hand/lesson.ipynb)

Read the [README](README.md) and do the six-value table on paper first. Every number below should match.

## 1. Six order values, plain Python

In [1]:
x = [10, 20, 20, 30, 40, 60]
n = len(x)

mean = sum(x) / n
sorted_x = sorted(x)
median = (sorted_x[n//2 - 1] + sorted_x[n//2]) / 2 if n % 2 == 0 else sorted_x[n//2]
print("mean  :", mean)
print("median:", median)

mean  : 30.0
median: 25.0


In [2]:
print(f"{'x':>3} {'x-mean':>7} {'(x-mean)²':>10}")
ss = 0
for xi in x:
    d = xi - mean
    ss += d * d
    print(f"{xi:>3} {d:>7.0f} {d*d:>10.0f}")
print(f"{'sum':>11} {ss:>10.0f}")

variance = ss / (n - 1)
std = variance ** 0.5
print(f"\nvariance = {ss:.0f} / {n-1} = {variance:.1f}")
print(f"std      = sqrt({variance:.1f}) = {std:.2f}")

  x  x-mean  (x-mean)²
 10     -20        400
 20     -10        100
 20     -10        100
 30       0          0
 40      10        100
 60      30        900
        sum       1600

variance = 1600 / 5 = 320.0
std      = sqrt(320.0) = 17.89


Mean 30, median 25, sum of squares 1600, variance 320, standard deviation 17.9: the README table.

## 2. Quartiles, IQR and the outlier fence

Median-of-halves method first (what you did on paper), then Pandas.

In [3]:
lower, upper = sorted_x[:n//2], sorted_x[n//2:]
q1 = sorted(lower)[len(lower)//2]
q3 = sorted(upper)[len(upper)//2]
iqr = q3 - q1
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, fences=({q1 - 1.5*iqr}, {q3 + 1.5*iqr})")

Q1=20, Q3=40, IQR=20, fences=(-10.0, 70.0)


In [4]:
import pandas as pd
s = pd.Series(x)
print(s.quantile([.25, .5, .75]))
print()
print(s.describe().round(2))

0.25    20.0
0.50    25.0
0.75    37.5
dtype: float64

count     6.00
mean     30.00
std      17.89
min      10.00
25%      20.00
50%      25.00
75%      37.50
max      60.00
dtype: float64


Pandas interpolates between sorted values, so its Q1 is 20.0 and Q3 is 37.5 instead of 40 on this tiny sample. On thousands of rows the methods agree. The *idea* (the middle half of the data) is what matters.

### Add one £150 order and watch what moves

In [5]:
s2 = pd.Series(x + [150])
q1, q3 = s2.quantile(.25), s2.quantile(.75)
fence = q3 + 1.5 * (q3 - q1)
print(f"mean   {s.mean():.1f} → {s2.mean():.1f}")
print(f"median {s.median():.1f} → {s2.median():.1f}")
print(f"std    {s.std():.1f} → {s2.std():.1f}")
print(f"upper fence {fence:.1f};  outliers: {list(s2[s2 > fence])}")

mean   30.0 → 47.1
median 25.0 → 30.0
std    17.9 → 48.2
upper fence 95.0;  outliers: [150]


The mean jumped by more than half; the median barely moved. The new order is flagged by Tukey's rule.

## 3. Correlation by hand

The five customers from lesson 02.2.

In [6]:
vx = [2, 4, 6, 8, 10]
vy = [50, 80, 90, 130, 150]
mx, my = sum(vx)/5, sum(vy)/5
sxy = sum((a-mx)*(b-my) for a, b in zip(vx, vy))
sxx = sum((a-mx)**2 for a in vx)
syy = sum((b-my)**2 for b in vy)
r = sxy / (sxx**0.5 * syy**0.5)
print(f"Sxy={sxy}, Sxx={sxx}, Syy={syy}")
print(f"r = {sxy} / (sqrt({sxx}) * sqrt({syy})) = {r:.3f}")
print("pandas:", pd.Series(vx).corr(pd.Series(vy)).round(3))

Sxy=500.0, Sxx=40.0, Syy=6400.0
r = 500.0 / (sqrt(40.0) * sqrt(6400.0)) = 0.988
pandas: 0.988


$r = 0.988$. Note the numerator and $S_{xx}$ are the same sums you used for the regression slope in 02.2; the slope was $500/40 = 12.5$, and correlation is that same relationship rescaled to lie between −1 and 1.

## 4. The real data

In [7]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

(541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [8]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

541,910 rows → 392,693 after cleaning


In [9]:
order_values = sales.groupby("Invoice")["Revenue"].sum()
customers = sales.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"))
customers["avg_order"] = customers["spend"] / customers["orders"]

### A typical order: mean vs median

In [10]:
print(f"mean order value:   £{order_values.mean():,.0f}")
print(f"median order value: £{order_values.median():,.0f}")
print(f"std:                £{order_values.std():,.0f}")
print(f"largest order:      £{order_values.max():,.0f}")
order_values.describe().round(0)

mean order value:   £480
median order value: £303
std:                £1,678
largest order:      £168,470


count     18532.0
mean        480.0
std        1678.0
min           0.0
25%         157.0
50%         303.0
75%         470.0
max      168470.0
Name: Revenue, dtype: float64

The mean is well above the median and the standard deviation is bigger than the mean itself. Both say the same thing: a heavy right tail. "Typical order" means the median here.

### Tukey's rule on real orders

In [11]:
q1, q3 = order_values.quantile([.25, .75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
flagged = order_values[order_values > upper]
print(f"Q1=£{q1:.0f}, Q3=£{q3:.0f}, IQR=£{iqr:.0f}, upper fence=£{upper:.0f}")
print(f"orders above the fence: {len(flagged):,} of {len(order_values):,} ({len(flagged)/len(order_values):.1%})")
print(f"revenue in those orders: {flagged.sum()/order_values.sum():.0%} of total")

Q1=£157, Q3=£470, IQR=£312, upper fence=£938
orders above the fence: 1,469 of 18,532 (7.9%)
revenue in those orders: 42% of total


Eight percent of orders are "outliers" by the rule, and they carry a large share of revenue. They are wholesale customers, not mistakes. The rule flags; you decide.

### Correlation matrix

In [12]:
corr = customers.corr()
corr.round(2)

,orders,spend,products,items,avg_order
orders,1.00,0.55,0.69,0.56,0.02
spend,0.55,1.00,0.39,0.92,0.39
products,0.69,0.39,1.00,0.41,0.04
items,0.56,0.92,0.41,1.00,0.41
avg_order,0.02,0.39,0.04,0.41,1.00


In [13]:
print("r(orders, spend) all customers:        ", customers["orders"].corr(customers["spend"]).round(3))
cut = customers["spend"].quantile(.99)
trimmed = customers[customers["spend"] < cut]
print("r(orders, spend) without top 1% spenders:", trimmed["orders"].corr(trimmed["spend"]).round(3))

r(orders, spend) all customers:         0.553
r(orders, spend) without top 1% spenders: 0.754


Removing 1% of customers changes $r$ noticeably: a handful of giants pull the straight line. Always compute correlation with a scatter plot next to it.

### `describe()` agrees with your formulas

In [14]:
import numpy as np
v = order_values.values
print("mean   by hand:", round(v.sum()/len(v), 2), "  pandas:", round(order_values.mean(), 2))
print("std    by hand:", round(np.sqrt(((v - v.mean())**2).sum()/(len(v)-1)), 2), "  pandas:", round(order_values.std(), 2))
print("median by hand:", np.sort(v)[len(v)//2] if len(v) % 2 else (np.sort(v)[len(v)//2-1] + np.sort(v)[len(v)//2])/2, "  pandas:", order_values.median())

mean   by hand: 479.56   pandas: 479.56
std    by hand: 1678.08   pandas: 1678.08
median by hand: 302.575   pandas: 302.575


## Quiz answers

1. When the distribution is skewed or has outliers. Report the median (and say so), or both.
2. Variance averages *squared* deviations, so its units are pounds squared; the square root brings it back to pounds.
3. The middle 50% of values span a range of £200.
4. Customers who ordered longer ago spend much less next month, in a close-to-straight-line way.
5. A U-shape (or any curve): e.g. spend is high for very new and very old customers and low in between. $r$ near 0, strong relationship.

**Next:** [01.9 The EDA checklist](../09-eda-checklist/)